# Modelo-07 — Processamento para o CatBoostClassifier

## Objetivo

Este notebook transforma a tabela mestra diária do sistema PSD em uma base pronta para modelagem. A unidade permanece **estação × plataforma × dia**, identificada por `id_psd` e `data_referencia`.

Nesta etapa serão realizados:

1. carregamento e validação da tabela mestra;
2. criação do alvo de ocorrência de falha nos próximos 7 dias;
3. criação de features históricas orientadas pelas hipóteses da Sprint 1;
4. aplicação de deslocamentos temporais para evitar vazamento do alvo;
5. definição da separação temporal entre treino, validação e teste;
6. exportação da base processada para o notebook de treinamento.

> A coluna `qtd_falhas_confirmadas` será utilizada para construir o alvo. Ela não poderá ser usada diretamente como feature contemporânea do modelo.

## Momento da previsão e janela do alvo

A previsão será realizada **ao final do dia de referência `D`**. Portanto:

- as features poderão utilizar informações conhecidas até o encerramento de `D`;
- o alvo será igual a `1` quando houver ao menos uma falha confirmada entre `D+1` e `D+7`;
- o alvo será igual a `0` quando não houver falha confirmada nessa janela futura;
- falhas ocorridas no próprio dia `D` não farão parte do alvo daquela linha;
- nenhuma informação registrada após `D` poderá ser utilizada como feature.

Representação da janela:

```text
Histórico disponível ───────── D │ D+1  D+2  D+3  D+4  D+5  D+6  D+7
                                │ └──────── janela do alvo ────────┘
                         previsão ao final de D
```

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")

print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Semente: {RANDOM_SEED}")

Pandas: 3.0.5
NumPy: 2.5.1
Semente: 42


In [2]:
CAMINHO_TABELA_MESTRA = Path("../../data/processados/modelo-07/tabela_mestra_psd.csv")

tabela_mestra = pd.read_csv(
    CAMINHO_TABELA_MESTRA,
    parse_dates=["data_referencia"],
)

print(f"Linhas: {tabela_mestra.shape[0]:,}")
print(f"Colunas: {tabela_mestra.shape[1]}")
display(tabela_mestra.head())

Linhas: 24,772
Colunas: 26


,data_referencia,id_psd,estacao_padrao,plataforma_padrao,qtd_alarmes_total,qtd_anomalias_criticas_30s,qtd_criticas_pico_manha,qtd_criticas_pico_tarde,duracao_total_alarmes_seg,duracao_media_alarmes_seg,duracao_maxima_alarme_seg,qtd_portas_com_alarme,qtd_corretivas_emergenciais,qtd_falhas_confirmadas,qtd_indisponibilidades,duracao_total_corretivas_horas,duracao_maxima_corretiva_horas,qtd_corretivas_programadas,qtd_indisponibilidades_programadas,duracao_total_programadas_horas,duracao_maxima_programada_horas,qtd_preventivas,qtd_preventivas_canceladas,qtd_preventivas_nao_canceladas,duracao_total_preventivas_horas,duracao_maxima_preventiva_horas
0,2023-02-27,BUT_P1,BUT,1,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,0,0,0,0.00,0.00
1,2023-02-27,BUT_P2,BUT,2,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,0,0,0,0.00,0.00
2,2023-02-27,FAL_P1,FAL,1,0,0,0,0,0.00,0.00,0.00,0,1,1,1,0.98,0.98,0,0,0.00,0.00,0,0,0,0.00,0.00
3,2023-02-27,FAL_P2,FAL,2,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,0,0,0,0.00,0.00
4,2023-02-27,FRA_P1,FRA,1,0,0,0,0,0.00,0.00,0.00,0,0,0,0,0.00,0.00,0,0,0.00,0.00,0,0,0,0.00,0.00
